# Imports

In [1]:
import os
from typing import Dict, List, Tuple, Any
from pathlib import Path
import pandas as pd
import numpy as np
from neuroHarmonize import harmonizationLearn, harmonizationApply
from collections import OrderedDict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from lifelines import CoxPHFitter

from numpy.typing import ArrayLike
from sksurv.util import Surv
from sksurv.metrics import (
    cumulative_dynamic_auc,
    concordance_index_censored,
    concordance_index_ipcw,
)

from types import SimpleNamespace

# Utils

In [10]:
def _missing_mask(series: pd.Series, nan_token: str = "Unknown") -> pd.Series:
    return series.eq(nan_token)

def impute_fit(
    df: pd.DataFrame,
    *,
    feature_cols: List[str],
    target_cols: List[str],
    nan_token: str = "Unknown",
    max_iter: int = 3,
    random_state: int = 42,
    verbose: bool = False,
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, Any]]]:
    """
    Fit per-column RandomForest imputers on TRAIN and return:
      - the TRAIN-imputed DataFrame
      - a dict of artefacts to apply on TEST (no refit)

    Iterative: >1 pass is useful when some predictors in `feature_cols` are also in `target_cols`
    (i.e., they are being imputed too). If all predictors are fully observed, set max_iter=1.
    """
    df_imp = df.copy()

    # Deterministic order over columns
    target_cols = list(OrderedDict.fromkeys(sorted(target_cols)))

    # Track which cells were originally missing; never touch originally observed values
    orig_missing: Dict[str, pd.Series] = {col: _missing_mask(df_imp[col], nan_token) for col in target_cols}

    # Prepare container for models and fallbacks
    models: Dict[str, Dict[str, Any]] = {col: {} for col in target_cols}

    def _fit_one(col: str) -> None:
        feats = [f for f in feature_cols if f != col]
        # rows where target is observed
        rows = ~_missing_mask(df_imp[col], nan_token)

        # Make sure all predictors needed for fitting are currently available
        for f in feats:
            rows &= ~_missing_mask(df_imp[f], nan_token)

        # Fallback (mode on TRAIN observed targets)
        mode = df_imp.loc[~_missing_mask(df_imp[col], nan_token), col].mode()
        fallback = mode.iloc[0] if not mode.empty else None

        if rows.sum() == 0:
            models[col] = {
                "model": None,
                "dummy_cols": None,
                "feature_cols": feats,
                "fallback": fallback,
                "nan_token": nan_token,
            }
            return

        # Xtr = pd.get_dummies(df_imp.loc[rows, feats])
        Xtr_source = df_imp.loc[rows, feats].copy() # new

        # For each categorical column, drop unused categories NOW
        for f in Xtr_source.select_dtypes(include="category").columns: # new
            Xtr_source[f] = Xtr_source[f].cat.remove_unused_categories() # new

        Xtr = pd.get_dummies(Xtr_source)        
        ytr = df_imp.loc[rows, col]
        # ytr = ytr.astype(int) # new

        rf = RandomForestClassifier(random_state=random_state)
        rf.fit(Xtr, ytr)

        models[col] = {
            "model": rf,
            "dummy_cols": Xtr.columns,
            "feature_cols": feats,
            "fallback": fallback,
            "nan_token": nan_token,
        }

    def _predict_one(col: str) -> int:
        """Predict for eligible, originally-missing cells. Returns number of newly imputed cells."""
        art = models[col]
        rf = art["model"]
        feats = art["feature_cols"]
        if rf is None:
            return 0

        # Eligible = originally missing & predictors currently available
        eligible = orig_missing[col].copy()
        for f in feats:
            eligible &= ~_missing_mask(df_imp[f], nan_token)

        if not eligible.any():
            return 0

        Xte = pd.get_dummies(df_imp.loc[eligible, feats])
        Xte = Xte.reindex(columns=art["dummy_cols"], fill_value=0)
        preds = rf.predict(Xte)
        df_imp.loc[eligible, col] = preds
        return int(eligible.sum())

    prev_remaining = None
    for it in range(max_iter):
        # 1) fit all column models on current df_imp (targets observed; predictors may include imputed values)
        for col in target_cols:
            _fit_one(col)

        # 2) apply one chained prediction pass
        new_fills = 0
        for col in target_cols:
            new_fills += _predict_one(col)

        # 3) check convergence by counting remaining originally-missing cells
        remaining = {col: int(_missing_mask(df_imp[col], nan_token)[orig_missing[col]].sum()) for col in target_cols}
        if verbose:
            print(f"[fit pass {it+1}] new_fills={new_fills} remaining={remaining}")

        if remaining == prev_remaining or new_fills == 0:
            break
        prev_remaining = remaining

    # Final TRAIN fallback for any cells that never became eligible
    for col in target_cols:
        leftover = orig_missing[col] & _missing_mask(df_imp[col], nan_token)
        if leftover.any():
            fb = models[col]["fallback"]
            if fb is not None:
                df_imp.loc[leftover, col] = fb
            # if fb is None, leave as nan_token (means column had no observed values)

    return df_imp, models

def impute_apply(
    df: pd.DataFrame,
    models: Dict[str, Dict[str, Any]],
    *,
    max_iter: int = 2,
) -> pd.DataFrame:
    """
    Apply previously fitted imputers on TEST with optional chained passes.
    Never refits; only overwrites cells that were originally missing in TEST.
    """
    df_imp = df.copy()

    # Determine original missing mask on the TEST data as of entry
    nan_token = next(iter(models.values()))["nan_token"] if models else "Unknown"
    target_cols = list(OrderedDict.fromkeys(sorted(models.keys())))
    orig_missing: Dict[str, pd.Series] = {col: _missing_mask(df_imp[col], nan_token) for col in target_cols}

    def _predict_one(col: str) -> int:
        art = models[col]
        rf = art["model"]
        feats = art["feature_cols"]
        if rf is None:
            return 0

        eligible = orig_missing[col].copy()
        for f in feats:
            eligible &= ~_missing_mask(df_imp[f], nan_token)

        if not eligible.any():
            return 0

        Xte = pd.get_dummies(df_imp.loc[eligible, feats])
        Xte = Xte.reindex(columns=art["dummy_cols"], fill_value=0)
        preds = rf.predict(Xte)
        df_imp.loc[eligible, col] = preds
        return int(eligible.sum())

    prev_remaining = None
    for _ in range(max_iter):
        new_fills = 0
        for col in target_cols:
            new_fills += _predict_one(col)

        remaining = {col: int(_missing_mask(df_imp[col], nan_token)[orig_missing[col]].sum()) for col in target_cols}
        if remaining == prev_remaining or new_fills == 0:
            break
        prev_remaining = remaining

    # Final TEST fallback using TRAIN fallback values
    for col in target_cols:
        leftover = orig_missing[col] & _missing_mask(df_imp[col], nan_token)
        if leftover.any():
            fb = models[col]["fallback"]
            if fb is not None:
                df_imp.loc[leftover, col] = fb
            # else: leave as nan_token

    return df_imp


def dummify(data, time_col, event_col, fixed_cols, new_cols, base_levels):
    df_cox = data[new_cols + fixed_cols + [event_col, time_col]]
    cat_columns = df_cox.select_dtypes(include=['object', 'category']).columns
    remaining_columns = list(set(fixed_cols+new_cols).difference(cat_columns))

    data_multi_cox = pd.get_dummies(df_cox, columns=cat_columns, drop_first=False)
    
    columns = []

    for col in cat_columns:
        if "Sex" in col:
            columns.append("Sex_Woman")

        elif any(col in new_col for new_col in new_cols): 
            for new_col, base_level in zip(new_cols, base_levels):
                if data[new_col].dtype not in ["object", "category"]:
                    continue

                new_col_levels = data[new_col].unique()
                for new_col_level in new_col_levels:
                    if new_col_level == base_level:
                        continue

                    columns.append(f"{new_col}_{new_col_level}")
        elif col == "KVP":
            columns.append(f"{col}_80.0")
            columns.append(f"{col}_120.0")         
        elif col == "Age_bin":
            columns.append(f"{col}_51–60")
            columns.append(f"{col}_>60")
        else:
            columns.append(f"{col}_Yes")
            
    columns += [time_col, event_col] + remaining_columns

    try:
        existing_columns = [c for c in columns if c in data_multi_cox.columns]
        df_cox = data_multi_cox[existing_columns]
    except Exception as e:
        print(e)
        print(f"remaining columns")
        print(columns)
        print(data_multi_cox.columns)
        
    
    return df_cox     

def get_cols_with_unk(data, cols):
    cols_with_unk = []
    for col in cols:
        count = (data[col] == "Unknown").sum()
        if count > 0:
            cols_with_unk.append(col)
    return cols_with_unk

def fit_cox_model_artifacts(
    df: pd.DataFrame,
    features: list,                 # radiomics columns
    clinical_features: list,        # optional clinical covariates (can be empty list)
    duration_col: str,
    event_col: str,
    site_col: str = "KVP",
    harmonize_by_site: bool = True,
    penalizer: float = 0.15,
    prediction_time: float = 48.0,
    strata: List = [],
    random_state: int = 42,

):
    """
    Fits a Cox model with radiomics predictors (standardized, optionally ComBat-harmonized)
    and optionally adjusts for clinical covariates (imputed and one-hot encoded as needed).

    Returns:
      cph: fitted CoxPHFitter
      hr_table: HR, 95% CI, p-values (for ALL covariates included in the model)
      X_model: final design matrix used to fit the model
    """
    np.random.seed(random_state)
    df_imp = df.copy()

    # ---- 1) Targets
    y_time  = df_imp[duration_col].astype(float).copy()
    y_event = df_imp[event_col].astype(bool).copy()
    
    # ---- 2) Imputation: categorical Unknowns + numeric NaNs
    if clinical_features is None:
        clinical_features = []
    all_to_check = list(dict.fromkeys(list(features) + list(clinical_features)))
    # print(f"all to check: {all_to_check}")
    
    # Split predictors
    numeric_with_nans = [
        col for col in all_to_check
        if pd.api.types.is_numeric_dtype(df_imp[col]) and df_imp[col].isna().any()
    ]
    # print(f"Numeric with nans: {numeric_with_nans}")
    categorical_fixed = [c for c in all_to_check if c not in numeric_with_nans and not pd.api.types.is_numeric_dtype(df_imp[c])]
    # print(f"categorical fixed: {categorical_fixed}")
    
    cols_with_unk = get_cols_with_unk(df_imp, categorical_fixed)
    # print(f"colswithunk: {cols_with_unk}")
    
    cat_imputer_models = None
    if len(cols_with_unk) > 0:
        # Build the frame passed to your imputer (predictors you allow inside the imputer)
        imputer_frame = df_imp[categorical_fixed].copy()
        # print((imputer_frame[categorical_fixed] == "Unknown").sum())
        clin_imp_block, cat_imputer_models = impute_fit(
            imputer_frame,
            feature_cols=categorical_fixed,
            target_cols=cols_with_unk,
            max_iter=3,
            random_state=random_state,
            verbose=False
        )
        # write back only imputed columns
        for c in cols_with_unk:
            df_imp[c] = clin_imp_block[c].values
    
    for f in df_imp.select_dtypes(include="category").columns:
        df_imp[f] = df_imp[f].cat.remove_unused_categories()
                
    for col in clinical_features:
        if pd.api.types.is_integer_dtype(df_imp[col]) and df_imp[col].nunique() <= 10:
            # print(f"here? {col}")
            df_imp[col] = df_imp[col].astype("category")

    X_rad = df_imp[features].copy()
    
    if len(clinical_features) > 0:
        X_model = dummify(
            data=df_imp,
            time_col=duration_col,
            event_col=event_col,
            fixed_cols=clinical_features, # fixed cols are to dummify
            new_cols=features,
            base_levels=[""]*len(clinical_features)
        )
        
        X_model[features] = X_rad[features]
    else:
        X_model = X_rad


    num_imputer = IterativeImputer(random_state=random_state, max_iter=10)
    num_imputer.fit(X_model)

    if X_model.isna().any().any():
        X_model.loc[:, :] = num_imputer.transform(X_model)
            
    imputer_columns = X_model.columns.tolist()
    
    harmonizer_model = None
    if harmonize_by_site and (site_col in df_imp.columns) and df_imp[site_col].nunique() >= 2:
        site = df_imp[site_col].astype(str).to_frame(name="SITE")
        harmonizer_model, X_h = harmonizationLearn(X_model[features].values, site)
        # print(X_h)
        X_rad_harmo = pd.DataFrame(X_h, index=X_model.index, columns=features)
        print(f"X_model columns: {X_model.columns}")
        X_model[features] = X_rad_harmo[features]
        
    num_cols = X_model.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude duration/event if present (shouldn't be in X_model, but safe)
    for col in (duration_col, event_col):
        if col in num_cols:
            num_cols.remove(col)    
                
    binary_cols = []
    for c in list(num_cols):
        u = pd.unique(X_model[c].dropna())
        if len(u) <= 2 and set(u).issubset({0, 1}):
            binary_cols.append(c)
        
    X_model_std = X_model.copy()    
    scale_cols = [c for c in num_cols if c not in binary_cols]
    scaler = None
    if len(scale_cols) > 0:
        scaler = StandardScaler(with_mean=True, with_std=True)
        X_model_std.loc[:, scale_cols] = scaler.fit_transform(X_model[scale_cols])
    
    final_columns = X_model_std.columns.tolist()  # NEW
        
    # ---- Build modeling DataFrame and fit Cox
    data_model = X_model_std.copy()
    data_model[duration_col] = y_time.values
    data_model[event_col] = y_event.values

    cph = CoxPHFitter(penalizer=penalizer)
    if strata:
        cph.fit(data_model, duration_col=duration_col, event_col=event_col,
                strata=strata)
    else:
        cph.fit(data_model, duration_col=duration_col, event_col=event_col)
        
    cph.check_assumptions(data_model, p_value_threshold=0.05, show_plots=False)

    # ---- 8) HR table
    summ = cph.summary.loc[:, ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].copy()
    summ = summ.rename(columns={
        "exp(coef)": "HR",
        "exp(coef) lower 95%": "CI_lower",
        "exp(coef) upper 95%": "CI_upper",
        "p": "p_value"
    })
    summ.index.name = "feature"
    hr_table = summ.reset_index()
    
    risks = 1 - cph.predict_survival_function(data_model).loc[prediction_time]
    signature_score = (data_model[features] * cph.params_).sum(axis=1)
    
    artifacts = {
        "features": features,
        "clinical_features": clinical_features,
        "duration_col": duration_col,
        "event_col": event_col,
        "site_col": site_col,
        "harmonize_by_site": harmonize_by_site,
        "cols_with_unk": cols_with_unk,
        "cat_imputer_models": cat_imputer_models,  
        "imputer_columns": imputer_columns,
        "num_imputer": num_imputer,
        "harmonizer_model": harmonizer_model,
        "scaler": scaler,
        "scale_cols": scale_cols,
        "final_columns": final_columns,
        "numeric_with_nans": numeric_with_nans,
    }    

    return cph, hr_table, (X_model, X_model_std), risks, signature_score, artifacts

In [11]:
def apply_pipeline_from_artifacts(
    df_B: pd.DataFrame,
    artifacts: dict,
    continuous_clinicals = {"Body mass index (BMI)", "Age at CCTA"}
) -> pd.DataFrame:
    """
    Replicates EXACTLY the preprocessing pipeline of fit_cox_model on an external dataset B.
    
    Handles:
    - Missing categories (e.g. "Smoker_Yes" missing in B) -> Adds column of 0s
    - Missing radiomics values (NaNs) -> Imputes using Training correlations
    """
    # Work on a copy
    df_imp = df_B.copy()

    # --- Unpack artifacts ---
    features          = artifacts["features"]
    clinical_features = artifacts["clinical_features"]
    duration_col      = artifacts["duration_col"]
    event_col         = artifacts["event_col"]
    site_col          = artifacts["site_col"]
    harmonize_by_site = artifacts["harmonize_by_site"]

    cols_with_unk     = artifacts["cols_with_unk"]
    cat_imputer_models = artifacts["cat_imputer_models"]
    num_imputer       = artifacts["num_imputer"]
    harmonizer_model  = artifacts["harmonizer_model"]
    scaler            = artifacts["scaler"]
    scale_cols        = artifacts["scale_cols"]
    final_columns     = artifacts["final_columns"]
    numeric_with_nans = artifacts["numeric_with_nans"]

    # ----------------------------------------------------------------
    # 0) SETUP SITE
    # ----------------------------------------------------------------
    if site_col in df_imp.columns:
        df_imp["SITE"] = df_imp[site_col].astype(str)
    else:
        df_imp["SITE"] = "0"

    # ----------------------------------------------------------------
    # 1) HARMONIZATION (ComBat)
    # ----------------------------------------------------------------
    if (harmonize_by_site and (harmonizer_model is not None) and len(features) > 0):
        covars = df_imp[["SITE"]].copy()
        covars["SITE"] = covars["SITE"].astype(float).astype(str)
        data = df_imp[features].to_numpy()
        
        # Handle Missing Training Sites ("Dummy Row Trick")
        train_site_labels = list(harmonizer_model["SITE_labels"]) 
        present = set(covars["SITE"].unique())
        missing = [lab for lab in train_site_labels if lab not in present]

        if missing:
            dummy_data = np.repeat(data[:1, :], len(missing), axis=0)
            dummy_covars = pd.DataFrame({"SITE": missing})
            covars_aug = pd.concat([covars, dummy_covars], ignore_index=True)
            data_aug = np.vstack([data, dummy_data])
            harmonized_aug = harmonizationApply(data_aug, covars_aug, harmonizer_model)
            harmonized = harmonized_aug[:-len(missing), :]
        else:
            harmonized = harmonizationApply(data, covars, harmonizer_model)

        df_imp.loc[:, features] = pd.DataFrame(harmonized, index=df_imp.index, columns=features)

    # ----------------------------------------------------------------
    # 2) CATEGORICAL IMPUTATION
    # ----------------------------------------------------------------
    all_to_check = list(dict.fromkeys(list(features) + list(clinical_features)))
    categorical_fixed = [c for c in all_to_check if c not in numeric_with_nans]

    if (cat_imputer_models is not None) and (len(cols_with_unk) > 0):
        imputer_frame_B = df_imp[categorical_fixed + features].copy()
        clin_imp_block_B = impute_apply(imputer_frame_B, cat_imputer_models, max_iter=2)
        
        for c in cols_with_unk:
            if c in clin_imp_block_B.columns:
                df_imp[c] = clin_imp_block_B[c].values

    for f in df_imp.select_dtypes(include="category").columns:
        df_imp[f] = df_imp[f].cat.remove_unused_categories()
        
    # for col in clinical_features:
    #     if col in df_imp.columns and pd.api.types.is_integer_dtype(df_imp[col]) and df_imp[col].nunique() <= 10:
    #         df_imp[col] = df_imp[col].astype("category")

    for col in clinical_features:
        if col in df_imp.columns:
            if col in continuous_clinicals:
                df_imp[col] = df_imp[col].astype(float)  # keep numeric
                continue
            if (pd.api.types.is_integer_dtype(df_imp[col]) and df_imp[col].nunique() <= 10):
                df_imp[col] = df_imp[col].astype("category")
            
    # ----------------------------------------------------------------
    # 3) DUMMIFY
    # ----------------------------------------------------------------
    # Produces X_model with currently available columns (e.g. 12 cols)
    if len(clinical_features) > 0:
        X_model = dummify(
            data=df_imp,
            time_col=duration_col,
            event_col=event_col,
            fixed_cols=clinical_features,
            new_cols=features,
            base_levels=[""] * len(clinical_features)
        )
        X_model[features] = df_imp[features]
    else:
        X_model = df_imp[features].copy()

    # ----------------------------------------------------------------
    # 4) ALIGN COLUMNS (The Fix)
    # ----------------------------------------------------------------
    # We must force X_model to match the exact shape expected by the Imputer (13 cols)
    # BEFORE we call transform().
    
    expected_cols = None
    
    # Try to get columns from the imputer object itself (sklearn > 1.0)
    if hasattr(num_imputer, "feature_names_in_"):
        expected_cols = num_imputer.feature_names_in_
    # Fallback 1: Did we save them in artifacts?
    elif "imputer_columns" in artifacts:
        expected_cols = artifacts["imputer_columns"]
    # Fallback 2: Assume final model columns = imputer columns (usually true)
    else:
        expected_cols = final_columns

    # CRITICAL STEP: Reindex
    # - Adds "Smoker_Yes" (filled with 0)
    # - Keeps Radiomics (preserving NaNs for now)
    # - Sorts columns to match Training order exactly
    if expected_cols is not None:
        X_model = X_model.reindex(columns=expected_cols, fill_value=0)

    # ----------------------------------------------------------------
    # 5) NUMERIC IMPUTATION
    # ----------------------------------------------------------------
    # Now that columns match (13 vs 13), we can safely impute the NaNs
    if (num_imputer is not None) and X_model.isna().any().any():
        # X_model has NaNs in radiomics. Imputer fills them using Clinicals + Other Radiomics.
        X_imputed_values = num_imputer.transform(X_model)
        X_model.loc[:, :] = X_imputed_values

    # ----------------------------------------------------------------
    # 6) SCALING
    # ----------------------------------------------------------------
    X_model_std = X_model.copy()
    
    if (scaler is not None) and (scale_cols is not None) and (len(scale_cols) > 0):
        # Safety check: ensure scale cols exist
        missing_scale = [c for c in scale_cols if c not in X_model_std.columns]
        if missing_scale:
             for c in missing_scale: X_model_std[c] = 0.0
        
        X_model_std.loc[:, scale_cols] = scaler.transform(X_model_std[scale_cols])

    # ----------------------------------------------------------------
    # 7) FINAL ALIGNMENT
    # ----------------------------------------------------------------
    # Ensure exact column order match with Cox Model
    X_final = X_model_std.reindex(columns=final_columns, fill_value=0)

    return X_model, X_final

In [15]:
def apply_cox_pipeline_from_artifacts_new_impute_radiomics(
    df_B: pd.DataFrame,
    artifacts: dict,
) -> pd.DataFrame:
    """
    Replicates EXACTLY the preprocessing pipeline of fit_cox_model on an external dataset B.
    
    Fixes applied:
    - Pipeline Order: Categorical Impute -> Dummify -> Numeric Impute -> Harmonize -> Scale.
    - Reindex Safety: Preserves NaNs in continuous columns for the IterativeImputer,
      only filling missing categorical dummies with 0.
    """
    # Work on a copy
    df_imp = df_B.copy()

    # --- Unpack artifacts ---
    features          = artifacts["features"]
    clinical_features = artifacts["clinical_features"]
    duration_col      = artifacts["duration_col"]
    event_col         = artifacts["event_col"]
    site_col          = artifacts["site_col"]
    harmonize_by_site = artifacts["harmonize_by_site"]

    cols_with_unk      = artifacts["cols_with_unk"]
    cat_imputer_models = artifacts["cat_imputer_models"]
    num_imputer        = artifacts["num_imputer"]
    harmonizer_model   = artifacts["harmonizer_model"]
    scaler             = artifacts["scaler"]
    scale_cols         = artifacts["scale_cols"]
    final_columns      = artifacts["final_columns"]
    numeric_with_nans  = artifacts["numeric_with_nans"]

    # ----------------------------------------------------------------
    # 0) SETUP SITE
    # ----------------------------------------------------------------
    if site_col in df_imp.columns:
        df_imp["SITE"] = df_imp[site_col].astype(str)
    else:
        df_imp["SITE"] = "0"

    # ----------------------------------------------------------------
    # 1) CATEGORICAL IMPUTATION (Moved up to match training)
    # ----------------------------------------------------------------
    all_to_check = list(dict.fromkeys(list(features) + list(clinical_features)))
    categorical_fixed = [c for c in all_to_check if c not in numeric_with_nans]

    if (cat_imputer_models is not None) and (len(cols_with_unk) > 0):
        # FIX: Only pass the exact categorical features the imputer expects
        imputer_frame_B = df_imp[categorical_fixed].copy()
        clin_imp_block_B = impute_apply(imputer_frame_B, cat_imputer_models, max_iter=2)
        
        for c in cols_with_unk:
            if c in clin_imp_block_B.columns:
                df_imp[c] = clin_imp_block_B[c].values

    for f in df_imp.select_dtypes(include="category").columns:
        df_imp[f] = df_imp[f].cat.remove_unused_categories()

    continuous_clinicals = {"Body mass index (BMI)", "Age at CCTA"}  # add others if needed

    for col in clinical_features:
        if col in df_imp.columns:
            if col in continuous_clinicals:
                df_imp[col] = df_imp[col].astype(float)  # keep numeric
                continue
            if (pd.api.types.is_integer_dtype(df_imp[col]) and df_imp[col].nunique() <= 10):
                df_imp[col] = df_imp[col].astype("category")
            
    # ----------------------------------------------------------------
    # 2) DUMMIFY
    # ----------------------------------------------------------------
    if len(clinical_features) > 0:
        X_model = dummify(
            data=df_imp,
            time_col=duration_col,
            event_col=event_col,
            fixed_cols=clinical_features,
            new_cols=features,
            base_levels=[""] * len(clinical_features)
        )
        X_model[features] = df_imp[features]
    else:
        X_model = df_imp[features].copy()

    # ----------------------------------------------------------------
    # 3) ALIGN COLUMNS (The NaN-Safe Fix)
    # ----------------------------------------------------------------
    expected_cols = None
    if hasattr(num_imputer, "feature_names_in_"):
        expected_cols = num_imputer.feature_names_in_
    elif "imputer_columns" in artifacts:
        expected_cols = artifacts["imputer_columns"]
    else:
        expected_cols = final_columns

    if expected_cols is not None:
        original_cols = X_model.columns.tolist()
        
        # CRITICAL FIX: Do not use fill_value=0 here globally. It destroys NaNs.
        X_model = X_model.reindex(columns=expected_cols)
        
        # Identify which columns were added because they were missing in B
        missing_cols = set(expected_cols) - set(original_cols)
        
        for c in missing_cols:
            # If the missing column is a continuous feature, leave it as NaN for the imputer.
            # If it's a dummy category (e.g., "Smoker_Yes"), fill it with 0.
            if (c not in features) and (scale_cols is None or c not in scale_cols):
                X_model[c] = X_model[c].fillna(0)

    # ----------------------------------------------------------------
    # 4) NUMERIC IMPUTATION
    # ----------------------------------------------------------------
    if (num_imputer is not None) and X_model.isna().any().any():
        X_imputed_values = num_imputer.transform(X_model)
        X_model.loc[:, :] = X_imputed_values

    # ----------------------------------------------------------------
    # 5) HARMONIZATION (ComBat) - Moved down to match training
    # ----------------------------------------------------------------
    if (harmonize_by_site and (harmonizer_model is not None) and len(features) > 0):
        covars = df_imp[["SITE"]].copy()
        covars["SITE"] = covars["SITE"].astype(float).astype(str)
        
        # Pull data directly from X_model so it benefits from the numeric imputation
        data = X_model[features].to_numpy()
        
        train_site_labels = list(harmonizer_model["SITE_labels"]) 
        present = set(covars["SITE"].unique())
        missing = [lab for lab in train_site_labels if lab not in present]

        if missing:
            dummy_data = np.repeat(data[:1, :], len(missing), axis=0)
            dummy_covars = pd.DataFrame({"SITE": missing})
            covars_aug = pd.concat([covars, dummy_covars], ignore_index=True)
            data_aug = np.vstack([data, dummy_data])
            harmonized_aug = harmonizationApply(data_aug, covars_aug, harmonizer_model)
            harmonized = harmonized_aug[:-len(missing), :]
        else:
            harmonized = harmonizationApply(data, covars, harmonizer_model)

        X_model.loc[:, features] = pd.DataFrame(harmonized, index=X_model.index, columns=features)

    # ----------------------------------------------------------------
    # 6) SCALING
    # ----------------------------------------------------------------
    X_model_std = X_model.copy()
    
    if (scaler is not None) and (scale_cols is not None) and (len(scale_cols) > 0):
        missing_scale = [c for c in scale_cols if c not in X_model_std.columns]
        if missing_scale:
             for c in missing_scale: X_model_std[c] = 0.0
        
        X_model_std.loc[:, scale_cols] = scaler.transform(X_model_std[scale_cols])

    # ----------------------------------------------------------------
    # 7) FINAL ALIGNMENT
    # ----------------------------------------------------------------
    X_final = X_model_std.reindex(columns=final_columns, fill_value=0)
    
    return X_model, X_final

In [12]:
def compute_survival_evaluation_metrics(
    outer_cv_data: Any,
    *,
    cap: float = 48.0,
    t_eval: float = 48.0,
    eps: float = 1e-9,
    invert_scores: bool = False,
    fallback_use_pooled_val_as_train: bool = True,
) -> Dict[str, ArrayLike]:
    """
    Compute survival evaluation metrics per outer-CV split.

    Parameters
    ----------
    outer_cv_data : object
        Must expose:
          - gts_val   : List[np.ndarray of shape (n_i, 2)] with columns [event, time]
          - probas_val: List[np.ndarray of shape (n_i,)]
        Optionally:
          - gts_train : List[np.ndarray of shape (m_i, 2)] (recommended for IPCW)

    cap : float
        Administrative censoring cutoff (e.g., 48.0). Test rows with time == cap will be nudged to cap-eps.

    t_eval : float
        Target time for time-dependent AUC; the function uses
        t_used = min(t_eval - eps, max(test_time) - eps, cap - eps, max(train_event_time) - eps).

    eps : float
        Small value to keep evaluation strictly below boundaries and for the nudge.

    invert_scores : bool
        If your model returns *higher = lower risk*, set True to flip the sign
        (all metrics assume higher = higher risk).

    fallback_use_pooled_val_as_train : bool
        If no `gts_train` is present, pool the other validation folds to form a
        training censoring KM for IPCW (leave-one-fold-out). If False, raises.

    Returns
    -------
    Dict[str, ArrayLike]
        Keys:
          - "tdAUC_at_t" : np.ndarray (n_splits,)   time-dependent AUC at t_used
          - "HarrellC"   : np.ndarray (n_splits,)   Harrell's concordance on test
          - "UnoC"       : np.ndarray (n_splits,)   Uno's IPCW concordance with tau_used
          - "t_used"     : np.ndarray (n_splits,)   the per-split AUC evaluation time
          - "tau_used"   : np.ndarray (n_splits,)   the per-split tau for Uno's C
        Splits that cannot be evaluated safely are filled with np.nan.
    """

    # ----- helpers -----
    def _to_surv(arr: np.ndarray) -> Surv:
        a = np.asarray(arr, dtype=np.float64)
        evt = a[:, 0].astype(bool)
        tim = a[:, 1].astype(float)
        return Surv.from_arrays(event=evt, time=tim)

    def _ensure_1d_float(x: ArrayLike) -> np.ndarray:
        s = np.asarray(x, dtype=float).reshape(-1)
        return s

    def _nudge_test_at_cap_down(arr: ArrayLike, time_col: int = 1,
                                cap: float = 48.0,
                                eps: float = 1e-9) -> np.ndarray:
        a = np.asarray(arr, dtype=np.float64).copy()
        t = a[:, time_col]
        mask = np.isclose(t, float(cap))
        if mask.any():
            a[mask, time_col] = float(cap) - float(eps)
        return a

    # ----- inputs -----
    gts_splits: List[np.ndarray]    = outer_cv_data.gts_val
    probas_splits: List[np.ndarray] = outer_cv_data.probas_val

    have_train = hasattr(outer_cv_data, "gts_train") and \
                 isinstance(outer_cv_data.gts_train, list) and \
                 len(outer_cv_data.gts_train) == len(gts_splits)

    n_splits = len(gts_splits)

    # ----- outputs (preallocate with NaN) -----
    tdAUC_at_t  = np.full(n_splits, np.nan, dtype=float)
    HarrellC    = np.full(n_splits, np.nan, dtype=float)
    UnoC        = np.full(n_splits, np.nan, dtype=float)
    t_used      = np.full(n_splits, np.nan, dtype=float)
    tau_used    = np.full(n_splits, np.nan, dtype=float)

    # ----- per-split loop -----
    for k, (gts_te_raw, scores_raw) in enumerate(zip(gts_splits, probas_splits)):

        # TRAIN data for IPCW:
        if have_train:
            gts_tr_raw = outer_cv_data.gts_train[k]
        else:
            if not fallback_use_pooled_val_as_train:
                # no training censoring KM available
                continue
            # Pool other folds as a proxy for training; leave out current test fold
            gts_tr_raw = np.vstack([g for i, g in enumerate(gts_splits) if i != k])

        # TRAIN: leave as-is; TEST: nudge any time == cap downward
        gts_tr = np.asarray(gts_tr_raw, dtype=np.float64)
        gts_te = _nudge_test_at_cap_down(gts_te_raw)

        # Build Surv
        Ytr = _to_surv(gts_tr)
        Yte = _to_surv(gts_te)

        # Scores
        scores = _ensure_1d_float(scores_raw)
        if invert_scores:
            scores = -scores

        # Safety: need at least one training event to define a sensible horizon
        tr_evt_times = Ytr["time"][Ytr["event"]]
        if tr_evt_times.size == 0:
            continue

        # Choose safe evaluation time for tdAUC and tau for Uno's C
        max_time_test = float(np.max(gts_te[:, 1]))
        max_time_train_event = float(np.max(tr_evt_times))

        # Strictly below every bound
        t_here = min(float(t_eval) - eps, max_time_test - eps, float(cap) - eps, max_time_train_event - eps)
        tau_here = min(max_time_test - eps, float(cap) - eps, max_time_train_event - eps)

        # Require positivity:
        if not (np.isfinite(t_here) and t_here > 0 and np.isfinite(tau_here) and tau_here > 0):
            continue

        # Compute Harrell's C on test (no tau needed)
        try:
            c_harrell, _, _, _, _ = concordance_index_censored(
                Yte["event"].astype(bool), Yte["time"].astype(float), scores
            )
            HarrellC[k] = float(c_harrell)
        except Exception as e:
            # leave as NaN
            print(e)
            pass

        # Compute Uno's C (IPCW) with tau_here (uses censoring KM from TRAIN)
        try:
            c_uno, _, _, _, _ = concordance_index_ipcw(Ytr, Yte, scores, tau=tau_here)
            UnoC[k] = float(c_uno)
            tau_used[k] = float(tau_here)
        except Exception:
            # leave as NaN
            pass

        # Compute tdAUC at a single time t_here
        try:
            auc_t, mean_auc = cumulative_dynamic_auc(Ytr, Yte, scores, times=[t_here])
            # auc_t is an array with one value; mean_auc equals that same value for a single time
            tdAUC_at_t[k] = float(auc_t[0])
            t_used[k] = float(t_here)
        except Exception as e:
            print(e)
            pass

    return {
        "tdAUC_at_t": tdAUC_at_t,   # per split
        "HarrellC": HarrellC,       # per split
        "UnoC": UnoC,               # per split
        "t_used": t_used,           # per split (for AUC)
        "tau_used": tau_used,       # per split (for Uno's C)
    }

# Fit

In [75]:
data_dir = Path("/Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/")
validation_root = data_dir / "external_validation/rebuttal"
validation_root.mkdir(parents=True, exist_ok=True)

df_derivation = pd.read_csv(data_dir / "output_no_MI_all_rad/analysis_checkpoints/20261103T104302Z_9f691260/df_der_no_hf_MEDIS_subset.csv")
df_validation = pd.read_csv(data_dir / "output_no_MI_all_rad/analysis_checkpoints/20261103T104302Z_9f691260/df_cpc_no_hf_MEDIS_subset.csv")

df_derivation_full = pd.read_csv(data_dir / "output_no_MI_all_rad/analysis_checkpoints/20261103T104302Z_9f691260/df_der_no_hf.csv")
df_validation_full = pd.read_csv(data_dir / "output_no_MI_all_rad/analysis_checkpoints/20261103T104302Z_9f691260/df_cpc_no_hf.csv")

/Users/francescopisu/Workspace/Research/Projects/CoroCTAiomics/working_solution/vmtkenv/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: DtypeWarning: Columns (0,4,30,35,58,64,104,105,106,108,131,2714,2721,2725,2845) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/francescopisu/Workspace/Research/Projects/CoroCTAiomics/working_solution/vmtkenv/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3553: DtypeWarning: Columns (4,138,2725) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [23]:
validation_root

PosixPath('/Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/external_validation/rebuttal')

In [21]:
import pickle 

signature = ["wavelet-LHL_gldm_LargeDependenceLowGrayLevelEmphasis_LAD",
    "wavelet-HHL_glcm_Correlation_LCX",
    "wavelet-LHH_firstorder_Skewness_RCA",
    "wavelet-LHL_firstorder_Skewness_RCA",
    "wavelet-LHL_firstorder_Mean_RCA",
    "original_glcm_Id_LAD",
    "wavelet-LHL_glcm_Idn_LAD",
    "wavelet-LLH_glcm_ClusterShade_LCX",
 ]


t_eval = 48.0
# clinical_columns = ['Age at CCTA','Sex','Smoker','Diabetes','Dyslipidemia','Hypertension', "Body mass index (BMI)"]
clinical_columns = ["Clinical Likelihood", "SS45"]

cph_model_rad_clin, hr_table_rad_clin, (X_rad_clin, X__rad_clin_std), risks_rad_clin, signature_score_rad_clin, artifacts = fit_cox_model_artifacts(
    df=df_derivation,
    features=signature,
    clinical_features = clinical_columns,
    duration_col="TIME",
    event_col="EVENT",
    site_col="KVP",            # used only for ComBat
    harmonize_by_site=True,    # set False if you want raw features
    prediction_time=t_eval,
    penalizer=0.15
)


# 1. Define the filename
output_filename = validation_root / "CoroCTAiomics_artifacts.pkl"

# 2. Create a bundle with everything needed for inference
#    We include t_eval so the inference script knows which timepoint to predict
pipeline_bundle = {
    "cph_model": cph_model_rad_clin,      # The trained Lifelines model
    "artifacts": artifacts,               # All preprocessing objects (Imputer, Scaler, ComBat)
    "t_eval": t_eval,                     # The prediction time (e.g., 48.0 months)
    "description": "Adjused Cox model"
}

# 3. Save to disk
with open(output_filename, "wb") as f:
    pickle.dump(pipeline_bundle, f)

print(f"✅ Success! Pipeline saved to: {output_filename}")
print(f"   Contains: {list(pipeline_bundle.keys())}")

X_model columns: Index(['SS45_Yes', 'TIME', 'EVENT', 'wavelet-HHL_glcm_Correlation_LCX',
       'wavelet-LHH_firstorder_Skewness_RCA', 'original_glcm_Id_LAD',
       'wavelet-LLH_glcm_ClusterShade_LCX',
       'wavelet-LHL_gldm_LargeDependenceLowGrayLevelEmphasis_LAD',
       'wavelet-LHL_firstorder_Skewness_RCA',
       'wavelet-LHL_firstorder_Mean_RCA', 'wavelet-LHL_glcm_Idn_LAD',
       'Clinical Likelihood'],
      dtype='object')
Proportional hazard assumption looks okay.
✅ Success! Pipeline saved to: /Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/external_validation/rebuttal/CoroCTAiomics_artifacts.pkl
   Contains: ['cph_model', 'artifacts', 't_eval', 'description']


# Predict

## Single patient

In [8]:
df_derivation.query("`Patient ID` == 96")[
    signature + clinical_columns + ["KVP", "EVENT", "TIME"]
].iloc[0].to_dict()

{'wavelet-LHL_gldm_LargeDependenceLowGrayLevelEmphasis_LAD': 0.002258470034351,
 'wavelet-HHL_glcm_Correlation_LCX': 0.0135131670361826,
 'wavelet-LHH_firstorder_Skewness_RCA': 0.1228842438194521,
 'wavelet-LHL_firstorder_Skewness_RCA': -0.361407389683918,
 'wavelet-LHL_firstorder_Mean_RCA': 6.486182401456452,
 'original_glcm_Id_LAD': 0.3048112394468715,
 'wavelet-LHL_glcm_Idn_LAD': 0.9693185280227522,
 'wavelet-LLH_glcm_ClusterShade_LCX': -625.1417922635238,
 'Clinical Likelihood': 19,
 'SS45': 'No',
 'KVP': 100.0,
 'EVENT': 1,
 'TIME': 6}

In [16]:
# new_patient_data = {
#     "wavelet-LHL_glcm_Idn_LAD": 0.92,
#     "wavelet-LLH_glrlm_RunEntropy_RCA": 4.84,
#     "wavelet-HLH_glszm_SizeZoneNonUniformityNormalized_LCX": 0.53,
#     "wavelet-HLL_glcm_Correlation_LCX": 0.34,
#     "Age at CCTA": .0,
#     "Sex": "Man",
#     "Smoker": "Yes",
#     "Diabetes": "Yes",
#     "Dyslipidemia": "Yes",
#     "Hypertension": "Yes",
#     "Body mass index (BMI)": 25,
#     "KVP": 100,
#     "TIME": None,
#     "EVENT": None
# }

new_patient_data = df_derivation.query("`Patient ID` == 96")[
    signature + clinical_columns + ["KVP", "EVENT", "TIME"]
].iloc[0].to_dict()

df_single = pd.DataFrame(new_patient_data, index=[0])

# --- 1. Load the artifacts ---
input_filename = validation_root / "CoroCTAiomics_artifacts.pkl"

print(f"Loading pipeline from {input_filename}...")
with open(input_filename, "rb") as f:
    loaded_bundle = pickle.load(f)

# Unpack the bundle
loaded_model = loaded_bundle["cph_model"]
loaded_artifacts = loaded_bundle["artifacts"]
loaded_t_eval = loaded_bundle["t_eval"]

print("Model and Artifacts loaded successfully.")


# --- 3. Apply the Pipeline ---
# NOTE: This function must be defined in your script!
X_external, X_external_std = apply_cox_pipeline_from_artifacts_new_impute_radiomics(
    df_single,
    loaded_artifacts  # Pass the loaded artifacts, not the original ones
)

# print(X_external["Body mass index (BMI)"])

# --- 4. Predict Risk ---
# Predict survival function
sf_external = loaded_model.predict_survival_function(X_external_std)

# Extract risk at the specific timepoint (48.0 months)
risk_external = 1.0 - sf_external.loc[loaded_t_eval].values

risk_percentage = risk_external[0] * 100
print(f"Risk of MACE at {loaded_t_eval} months: {risk_percentage:.2f}%")

Loading pipeline from /Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/external_validation/rebuttal/CoroCTAiomics_artifacts.pkl...
Model and Artifacts loaded successfully.
Risk of MACE at 48.0 months: 13.19%


## External dataset

In [22]:

# --- 1. Load the artifacts ---
input_filename = validation_root / "CoroCTAiomics_artifacts.pkl"

print(f"Loading pipeline from {input_filename}...")
with open(input_filename, "rb") as f:
    loaded_bundle = pickle.load(f)

# Unpack the bundle
loaded_model = loaded_bundle["cph_model"]
loaded_artifacts = loaded_bundle["artifacts"]
loaded_t_eval = loaded_bundle["t_eval"]

print("Model and Artifacts loaded successfully.")

# --- 3. Apply the Pipeline ---
X_external, X_external_std = apply_cox_pipeline_from_artifacts_new_impute_radiomics(
    df_validation,
    loaded_artifacts  # Pass the loaded artifacts, not the original ones
)

# --- 4. Predict Risk ---
# Predict survival function
sf_external = loaded_model.predict_survival_function(X_external_std)

# Extract risk at the specific timepoint (48.0 months)
risk_external = 1.0 - sf_external.loc[loaded_t_eval].values


# A. Ground Truth for EXTERNAL (Validation)
# Shape: (N_val, 2) -> [[Event, Time], ...]
gts_external = np.stack([
    df_validation["EVENT"].values.astype(bool), 
    df_validation["TIME"].values.astype(float)
], axis=1)

# B. Ground Truth for TRAINING (Used for IPCW Censoring Distribution)
# Shape: (N_train, 2) -> [[Event, Time], ...]
gts_train_arr = np.stack([
    df_derivation["EVENT"].values.astype(bool), 
    df_derivation["TIME"].values.astype(float)
], axis=1)

cv_data_wrapper = SimpleNamespace(
    gts_val    = [gts_external],     # List of [Array(N, 2)]
    probas_val = [risk_external.flatten()],  # List of [Array(N,)]
    gts_train  = [gts_train_arr]     # List of [Array(M, 2)]
)

# 4. Compute Metrics
metrics = compute_survival_evaluation_metrics(
    cv_data_wrapper,
    cap=48.0,              # Administrative censoring cap
    t_eval=loaded_t_eval,  # 48.0 months
    eps=1e-9,
    invert_scores=False,   # False because Higher Risk (1-S) = Shorter Survival
)

# 5. Extract and Print
# The function returns lists/arrays (one per fold). We access index [0].
print(f"\nExternal Validation Results:")
print(f"------------------------------------------------")
print(f"Timepoint:   {metrics['t_used'][0]:.1f} months")
print(f"Harrell's C: {metrics['HarrellC'][0]:.4f}")
print(f"Uno's C:     {metrics['UnoC'][0]:.4f} (IPCW using Train G(t))")
print(f"tdAUC:       {metrics['tdAUC_at_t'][0]:.4f}")
print(f"------------------------------------------------")

Loading pipeline from /Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/external_validation/rebuttal/CoroCTAiomics_artifacts.pkl...
Model and Artifacts loaded successfully.

External Validation Results:
------------------------------------------------
Timepoint:   48.0 months
Harrell's C: 0.7841
Uno's C:     0.7856 (IPCW using Train G(t))
tdAUC:       0.7944
------------------------------------------------


In [ ]:
# =============================================================================
# FIT AND SAVE THE PRI-ONLY MODEL USED FOR RISK-STRATUM ASSIGNMENT
# =============================================================================

with open(
    validation_root / "PRI_risk_strata_thresholds.json",
    "r",
) as file:
    pri_strata_config = json.load(file)

(
    pri_only_model,
    pri_only_hr_table,
    (
        pri_only_X,
        pri_only_X_std,
    ),
    pri_only_derivation_risks,
    pri_only_signature_score,
    pri_only_artifacts,
) = fit_cox_model_artifacts(
    df=df_derivation,
    features=signature,
    clinical_features=[],       # Crucial: PRI only
    duration_col="TIME",
    event_col="EVENT",
    site_col="KVP",
    harmonize_by_site=True,
    prediction_time=48.0,
    penalizer=0.15,             # Same predictive penalizer
)

pri_only_bundle = {
    "cph_model": pri_only_model,
    "artifacts": pri_only_artifacts,
    "t_eval": 48.0,
    "strata": pri_strata_config,
    "description": (
        "PRI-only model for PRI score and risk-stratum assignment"
    ),
}

pri_only_bundle_path = (
    validation_root / "CoroCTAiomics_PRI_only_artifacts.pkl"
)

with open(pri_only_bundle_path, "wb") as file:
    pickle.dump(pri_only_bundle, file)

print("Saved:", pri_only_bundle_path)

X_model columns: Index(['wavelet-LHL_gldm_LargeDependenceLowGrayLevelEmphasis_LAD',
       'wavelet-HHL_glcm_Correlation_LCX',
       'wavelet-LHH_firstorder_Skewness_RCA',
       'wavelet-LHL_firstorder_Skewness_RCA',
       'wavelet-LHL_firstorder_Mean_RCA', 'original_glcm_Id_LAD',
       'wavelet-LHL_glcm_Idn_LAD', 'wavelet-LLH_glcm_ClusterShade_LCX'],
      dtype='object')
Proportional hazard assumption looks okay.
Saved: /Volumes/X9Pro/output/CoroCTAiomics/analysis/v2/28072025/data/external_validation/rebuttal/CoroCTAiomics_PRI_only_artifacts.pkl


In [26]:
# =============================================================================
# PRI-ONLY PREDICTION AND EXACT STUDY STRATUM
# =============================================================================

with open(
    validation_root / "CoroCTAiomics_PRI_only_artifacts.pkl",
    "rb",
) as file:
    pri_bundle = pickle.load(file)

pri_model = pri_bundle["cph_model"]
pri_artifacts = pri_bundle["artifacts"]
pri_time = float(pri_bundle["t_eval"])

pri_q25 = float(pri_bundle["strata"]["q25"])
pri_q75 = float(pri_bundle["strata"]["q75"])

# df_single must contain the eight radiomic features and KVP.
_, X_single_pri_std = (
    apply_cox_pipeline_from_artifacts_new_impute_radiomics(
        df_single,
        pri_artifacts,
    )
)

pri_survival = pri_model.predict_survival_function(
    X_single_pri_std,
    times=[pri_time],
)

pri_only_risk = float(
    1.0 - pri_survival.iloc[0, 0]
)

# pd.cut reproduces the original notebook's boundary rules exactly:
#   Low:          score <= Q25
#   Intermediate: Q25 < score <= Q75
#   High:         score > Q75
pri_stratum = pd.cut(
    [pri_only_risk],
    bins=[
        -np.inf,
        pri_q25,
        pri_q75,
        np.inf,
    ],
    labels=[
        "Low (<25th percentile)",
        "Intermediate (25th–75th percentile)",
        "High (>75th percentile)",
    ],
    right=True,
)[0]

print(f"PRI-only 4-year risk score: {100 * pri_only_risk:.2f}%")
print(f"Derivation Q25 threshold: {100 * pri_q25:.2f}%")
print(f"Derivation Q75 threshold: {100 * pri_q75:.2f}%")
print(f"PRI risk stratum: {pri_stratum}")

PRI-only 4-year risk score: 12.10%
Derivation Q25 threshold: 6.45%
Derivation Q75 threshold: 10.57%
PRI risk stratum: High (>75th percentile)


In [33]:
pri_model.summary

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
wavelet-LHL_gldm_LargeDependenceLowGrayLevelEmphasis_LAD,-0.127070,0.880672,0.071180,-0.266581,0.012440,0.765994,1.012518,0.0,-1.785190,0.074231,3.751844
wavelet-HHL_glcm_Correlation_LCX,-0.095591,0.908835,0.064141,-0.221305,0.030123,0.801472,1.030581,0.0,-1.490333,0.136137,2.876871
wavelet-LHH_firstorder_Skewness_RCA,0.079034,1.082242,0.063316,-0.045063,0.203132,0.955938,1.225234,0.0,1.248254,0.211938,2.238286
wavelet-LHL_firstorder_Skewness_RCA,-0.106061,0.899370,0.063026,-0.229589,0.017467,0.794860,1.017621,0.0,-1.682816,0.092411,3.435797
wavelet-LHL_firstorder_Mean_RCA,-0.077483,0.925443,0.060324,-0.195715,0.040749,0.822246,1.041591,0.0,-1.284458,0.198982,2.329292
original_glcm_Id_LAD,-0.159994,0.852148,0.064748,-0.286899,-0.033090,0.750587,0.967452,0.0,-2.471017,0.013473,6.213790
wavelet-LHL_glcm_Idn_LAD,0.103921,1.109513,0.064847,-0.023177,0.231019,0.977090,1.259884,0.0,1.602560,0.109032,3.197178
wavelet-LLH_glcm_ClusterShade_LCX,-0.108806,0.896905,0.053175,-0.213028,-0.004584,0.808134,0.995427,0.0,-2.046164,0.040740,4.617403


In [30]:
df_derivation["Type of chest pain "].unique()

array(['Typycal Chest Pain', nan, 'Chest Pain At Rest', 'Not Specified',
       'Atypical Chest Pain'], dtype=object)

In [71]:
df_derivation[df_derivation["Patient ID"].isin([96, 814, 893])]

,Patient ID,Age at CCTA,Sex,Smoker,Diabetes,Dyslipidemia,Hypertension,Dyspnea,Type of chest pain,Familiarity for CAD,...,CADRADS,SS45,MAX STENOSIS,Clinical Likelihood,Clinical Likelihood Class,SEGMENT_SegmentPlaqueMeanIntensityMax_11,SEGMENT_SegmentMaximalPlaqueThickness_1,SEGMENT_SegmentMaximalPlaqueThickness_11,SEGMENT_SegmentPlaqueBurdenMean_1,SEGMENT_SegmentPlaqueVolume_5_6
83,96,78.0,Man,Yes,No,No,Yes,No,NaN,No,...,Score 2,No,35,19,Moderate,456.063291,1.89884,1.78924,0.242957,121.851600
591,814,25.0,Woman,No,No,No,No,No,NaN,No,...,Score 0,No,0,0,Very low,321.448276,2.64136,1.50249,0.350583,97.397226
609,893,83.0,Woman,Yes,No,Yes,Yes,Yes,NaN,No,...,Score 5,Yes,100,10,Low,548.626263,2.15755,2.29553,0.412541,508.006453


In [74]:
df_derivation.query("`Age at CCTA` == 83.0 and `Sex` == 'Man'")

,Patient ID,Age at CCTA,Sex,Smoker,Diabetes,Dyslipidemia,Hypertension,Dyspnea,Type of chest pain,Familiarity for CAD,...,CADRADS,SS45,MAX STENOSIS,Clinical Likelihood,Clinical Likelihood Class,SEGMENT_SegmentPlaqueMeanIntensityMax_11,SEGMENT_SegmentMaximalPlaqueThickness_1,SEGMENT_SegmentMaximalPlaqueThickness_11,SEGMENT_SegmentPlaqueBurdenMean_1,SEGMENT_SegmentPlaqueVolume_5_6
135,165,83.0,Man,Yes,No,Unknown,Yes,No,Atypical Chest Pain,No,...,Score 1,No,15,19,Moderate,254.752381,1.372990,2.10245,0.197608,131.065563
680,1061,83.0,Man,Yes,No,No,Yes,Yes,NaN,No,...,Score 1,No,10,27,Moderate,267.710843,0.947889,1.06611,0.260414,146.367738
997,2131,83.0,Man,Yes,Yes,Yes,Yes,No,Atypical Chest Pain,No,...,Score 4,Yes,70,24,Moderate,463.454545,1.765470,1.55076,0.336729,328.508065


In [ ]:
{"Age at CCTA": 42,
"Sex": "Man",
"Hypertension": "No",
"Dyslipdemia": "Yes",
"Smoker": "Yes",
"Diabetes": "No",
"Dyspnea": "No",
"Type of chest pain ": None,
"CADRADS": "Score 2"
}